# Dask Scaling Benchmarks

Adapted from [Dask Benchmarks blog post (2017)](https://blog.dask.org/2017/07/03/scaling).

Measures Dask distributed scheduler performance across:
- Computational and communication patterns (embarrassingly parallel, tree reduction, nearest neighbor, sequential)
- Task duration (fast / 100ms / 1s)
- APIs: raw tasks, `dask.array`, `dask.dataframe`

> **Note:** Original benchmarks used weak scaling on a Kubernetes cluster (1–256 two-core workers). This notebook runs locally; set `n_cores` and `n` below to match your environment.

## Setup

In [1]:
import math
import time
import os

import dask
import dask.array as da
import dask.dataframe as dd
from dask import delayed
from dask.distributed import Client, wait, as_completed
from dask_gateway import Gateway


In [2]:
gateway = Gateway()
clusters = gateway.list_clusters()

if clusters:
    cluster = gateway.connect(clusters[0].name)
    print(f"Connected to existing cluster: {clusters[0].name}")
else:
    cluster = gateway.new_cluster()
    cluster.adapt(minimum=2, maximum=10)
    print("Created new cluster")

Connected to existing cluster: staging.5266522a74ae499ca2eaf99806bab9b5


In [3]:
#Connect to the gateway cluster
client = cluster.get_client()
# client 

In [ ]:
# Scale parameters — original post used n_cores up to 512
# n_cores = len(client.ncores())          # number of worker processes
client.wait_for_workers(n_workers=2)

n_cores = sum(client.ncores().values())
n = n_cores                              # used as a multiplier in several benchmarks

print(f"n_cores = {n_cores}, n = {n}")

## Helper Functions

In [ ]:
def inc(x):
    return x + 1

def add(x, y):
    return x + y

def slowinc(x, delay=0.1):
    time.sleep(delay)
    return x + 1

def slowadd(x, y, delay=0.1):
    time.sleep(delay)
    return x + y

def slowsum(L, delay=0.1):
    time.sleep(delay)
    return sum(L)

---
## Part 1: Task Benchmarks

Raw task scheduling — the foundation for arrays and dataframes.

### 1.1 Embarrassingly Parallel

Each task is independent — maximum potential parallelism.

In [ ]:
# 1-second tasks
t0 = time.time()
futures = client.map(slowinc, range(4 * n), delay=1)
wait(futures)
elapsed = time.time() - t0
n_tasks = 4 * n
print(f"1s tasks  | {n_tasks} tasks | {elapsed:.2f}s | {n_tasks/elapsed:.0f} tasks/s")

In [ ]:
# 100ms tasks
t0 = time.time()
futures = client.map(slowinc, range(100 * n_cores))  # default delay=0.1
wait(futures)
elapsed = time.time() - t0
n_tasks = 100 * n_cores
print(f"100ms tasks | {n_tasks} tasks | {elapsed:.2f}s | {n_tasks/elapsed:.0f} tasks/s")

In [ ]:
# Fast (microsecond) tasks
t0 = time.time()
futures = client.map(inc, range(n_cores * 200))
wait(futures)
elapsed = time.time() - t0
n_tasks = n_cores * 200
print(f"Fast tasks  | {n_tasks} tasks | {elapsed:.2f}s | {n_tasks/elapsed:.0f} tasks/s")

**Expected:** Fast tasks saturate at ~2000–3000 tasks/s (scheduler-bound). Longer tasks scale with core count.

### 1.2 Tree Reduction

Combine neighboring elements recursively until one result remains. Tests task dependency handling.

In [ ]:
t0 = time.time()

L = list(range(2**7 * n))
while len(L) > 1:
    L = [delayed(slowadd)(a, b) for a, b in zip(L[::2], L[1::2])]

result = L[0].compute()
elapsed = time.time() - t0
print(f"Tree reduction | result={result} | {elapsed:.2f}s")

### 1.3 Nearest Neighbor

Each element depends on its neighbor — common in timeseries and sliding-window operations.

In [ ]:
t0 = time.time()

L = list(range(20 * n))
L = client.map(slowadd, L[:-1], L[1:])
L = client.map(slowadd, L[:-1], L[1:])
wait(L)

elapsed = time.time() - t0
print(f"Nearest neighbor | {elapsed:.2f}s")

### 1.4 Sequential (No Parallelism)

Each step depends on the previous — measures scheduler round-trip latency.

In [ ]:
# Disable task fusion so each step is a separate scheduler round-trip
with dask.config.set(optimization__fuse__active=False):
    t0 = time.time()
    x = 1
    for i in range(100):
        x = delayed(inc)(x)
    result = x.compute()
    elapsed = time.time() - t0

print(f"Sequential | 100 steps | result={result} | {elapsed:.2f}s | {100/elapsed:.0f} roundtrips/s")

**Expected:** ~100 roundtrips/s → ~10ms latency per step.

### 1.5 Client-in-the-Loop Reduction

Process futures as they complete, accumulating results. Offloads dependency tracking from the scheduler to the client.

In [ ]:
t0 = time.time()

futures = client.map(slowinc, range(n * 20))
pool = as_completed(futures)
batches = pool.batches()

while True:
    try:
        batch = next(batches)
        if len(batch) == 1:
            batch += next(batches)
    except StopIteration:
        break
    future = client.submit(slowsum, batch)
    pool.add(future)

elapsed = time.time() - t0
print(f"Client-in-loop reduction | {elapsed:.2f}s")

---
## Part 2: Array Benchmarks

`dask.array` wraps NumPy chunks. Chunk size is 2000×2000 as in the original post.

### 2.1 Create Dataset

In [ ]:
import numpy as np

N = int(5000 * math.sqrt(n_cores))
print(f"Array shape: ({N}, {N})")

t0 = time.time()
x = da.random.randint(0, 10000, size=(N, N), chunks=(2000, 2000))
x = x.persist()
wait(x)
elapsed = time.time() - t0
print(f"Create array | {elapsed:.2f}s")

### 2.2 Elementwise Computation

In [ ]:
t0 = time.time()
y = da.sin(x) ** 2 + da.cos(x) ** 2
y = y.persist()
wait(y)
elapsed = time.time() - t0

mb = x.nbytes / 1e6
print(f"Elementwise sin²+cos² | {elapsed:.2f}s | {mb/elapsed:.0f} MB/s")

### 2.3 Reduction

In [ ]:
t0 = time.time()
result = x.std().compute()
elapsed = time.time() - t0
print(f"std() reduction | result={result:.4f} | {elapsed:.2f}s")

### 2.4 Random Access

In [ ]:
t0 = time.time()
val = x[1234, 4567].compute()
elapsed = time.time() - t0
print(f"Random access x[1234,4567] | value={val} | {elapsed*1000:.1f} ms")

**Expected:** ~10–20ms. Doesn't improve with more workers.

### 2.5 Communication: Array + Transpose

In [ ]:
t0 = time.time()
y = x + x.T
y = y.persist()
wait(y)
elapsed = time.time() - t0

mb = x.nbytes / 1e6
print(f"x + x.T (communication) | {elapsed:.2f}s | {mb/elapsed:.0f} MB/s")

### 2.6 Rechunking (All-to-All Communication)

In [ ]:
t0 = time.time()
y = x.rechunk((20000, 200)).persist()
wait(y)
y = y.rechunk((200, 20000)).persist()
wait(y)
elapsed = time.time() - t0

mb = x.nbytes / 1e6
print(f"Rechunk round-trip | {elapsed:.2f}s | {mb/elapsed:.0f} MB/s")

### 2.7 Nearest Neighbor (map_overlap)

In [ ]:
t0 = time.time()
y = x.map_overlap(lambda b: b + 1, depth=1).persist()  # lightweight proxy for slowinc
wait(y)
elapsed = time.time() - t0
print(f"map_overlap (nearest neighbor) | {elapsed:.2f}s")

---
## Part 3: DataFrame Benchmarks

`dask.dataframe` wraps Pandas partitions.

### 3.1 Create Dataset

In [ ]:
N_rows = 2_000_000 * n_cores
print(f"DataFrame rows: {N_rows:,}")

t0 = time.time()
x_arr = da.random.randint(0, 10000, size=(N_rows, 10), chunks=(1_000_000, 10))
df = dd.from_dask_array(x_arr).persist()
wait(df)
elapsed = time.time() - t0
print(f"Create dataframe | {elapsed:.2f}s")
df.dtypes

### 3.2 Elementwise

In [ ]:
# map_partitions with a slow function
t0 = time.time()
y = df.map_partitions(slowinc, meta=df).persist()
wait(y)
elapsed = time.time() - t0
print(f"map_partitions slowinc | {elapsed:.2f}s")

In [ ]:
# Fast arithmetic
t0 = time.time()
y = (df[0] + 1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10).persist()
wait(y)
elapsed = time.time() - t0

mb = df.memory_usage(deep=True).sum().compute() / 1e6
print(f"Elementwise arithmetic | {elapsed:.2f}s | {mb/elapsed:.0f} MB/s")

### 3.3 Random Access

In [ ]:
t0 = time.time()
row = df.loc[123456].compute()
elapsed = time.time() - t0
print(f"df.loc[123456] | {elapsed*1000:.1f} ms")
row

### 3.4 Reductions

In [ ]:
t0 = time.time()
result = df.std().compute()
elapsed = time.time() - t0
print(f"df.std() (full) | {elapsed:.2f}s")

t0 = time.time()
result = df[0].std().compute()
elapsed = time.time() - t0
print(f"df[0].std() (single column) | {elapsed:.2f}s")

In [ ]:
# Groupby aggregation
t0 = time.time()
result = df.groupby(0)[1].mean().compute()
elapsed = time.time() - t0
print(f"groupby(0)[1].mean() | {elapsed:.2f}s")
result.head()

### 3.5 Shuffle (set_index / groupby-apply)

Requires full data movement — the most expensive operation.

In [ ]:
# groupby-apply (triggers shuffle)
t0 = time.time()
result = df.groupby(0).apply(len, meta=(None, 'int64')).compute()
elapsed = time.time() - t0
print(f"groupby(0).apply(len) | {elapsed:.2f}s")

In [ ]:
# set_index
t0 = time.time()
y = df.set_index(1).persist()
wait(y)
elapsed = time.time() - t0

mb = df.memory_usage(deep=True).sum().compute() / 1e6
print(f"set_index(1) | {elapsed:.2f}s | {mb/elapsed:.0f} MB/s")

### 3.6 Timeseries: Rolling Mean

In [ ]:
t0 = time.time()
y = df.rolling(5).mean().persist()
wait(y)
elapsed = time.time() - t0
print(f"rolling(5).mean() | {elapsed:.2f}s")

---
## Summary

Key takeaways from the original benchmarks:

| Finding | Detail |
|---|---|
| Scheduler cap | ~3000 tasks/s regardless of cluster size |
| Fast tasks don't scale | Overhead-bound; longer tasks benefit from more workers |
| Dependencies are cheap | Tree reduction and nearest neighbor scale like embarrassingly parallel |
| All-to-all is harder | Rechunk and shuffle scale sub-linearly |
| Bigger workers + chunks | 5–10× improvement; prefer 16-core workers and ~1 GB chunks |

### Expert configuration (from the post)
- **Workers:** fewer, larger (e.g., 32×16-core instead of 256×2-core)
- **Array chunks:** 10,000×10,000 instead of 2,000×2,000
- **DataFrame partitions:** 10M rows instead of 1M rows

In [ ]:
client.close()